# Reasoning Model Fine-Tuning with Unsloth and GRPO

In this session, we step away from orchestrating models and change the weights themselves. Using Unsloth's implementation of Group Relative Policy Optimization (GRPO) — the reinforcement learning algorithm behind DeepSeek-R1 — we will teach `meta-llama/Llama-3.2-3B-Instruct` to produce structured, step-by-step reasoning on grade-school math problems (GSM8K).

This is not a 1-to-1 reproduction of the DeepSeek-R1 paper, but it exercises the same core loop:

```text
prompt -> sample a group of 8 completions -> score each with reward functions
       -> compute group-relative advantage -> policy update (with KL penalty) -> repeat
```

Because each completion is scored relative to the *group average*, GRPO needs no separate value network (critic) — a big part of why it is practical at small scale. And unlike supervised fine-tuning, we never show the model *how* to reason: we only reward it when the reasoning format and final answer are right, and let it discover the rest.

We train a **16-bit LoRA adapter** (not QLoRA): GRPO spends most of its time generating completions, and vLLM's fast-generation path is fastest and most stable on 16-bit weights.

> **Hardware requirements:** this notebook trains locally and needs an NVIDIA GPU with compute capability 8.0+ (Ampere or newer — recent vLLM releases dropped T4/Turing support), 16 GB+ VRAM, on Linux or WSL2. See the session README for prerequisites and out-of-memory knobs.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain the GRPO training loop and how it differs from SFT and PPO-style RLHF.
- Load a base model with Unsloth and attach LoRA adapters for parameter-efficient training.
- Explain the trade-off between 16-bit LoRA and 4-bit QLoRA.
- Prepare a verifiable-answer dataset (GSM8K) for reward-based training.
- Design stacked reward functions that shape both format and correctness.
- Configure and run TRL's `GRPOTrainer` and interpret its reward logs.
- Compare base and fine-tuned behavior, and save and load the trained LoRA adapter.

## Table of Contents

- **Breakout Room #1: Model Loading and LoRA**
  - Task 1: Environment Setup
  - Task 2: Load the Base Model
  - Task 3: Attach LoRA Adapters
  - Question #1, Question #2, and Question #3
- **Breakout Room #2: GRPO Training on GSM8K**
  - Task 4: Prepare the GSM8K Dataset
  - Task 5: Define Reward Functions
  - Task 6: Configure GRPO
  - Question #4
  - Task 7: Train with GRPOTrainer
  - Task 8: Compare Before and After
  - Task 9: Save and Load the LoRA

---
# Breakout Room #1
## Model Loading and LoRA

We load the base model with Unsloth's vLLM-backed fast-inference path, then attach LoRA adapters so training touches only a small fraction of the weights.

## Task 1: Environment Setup

From the `15_Reasoning_Model_Fine_Tuning` folder, install dependencies with uv (Linux or WSL2 with an NVIDIA GPU — see the README for prerequisites):

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

`meta-llama/Llama-3.2-3B-Instruct` is a gated repository: accept the license on its [Hugging Face model page](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) and log in below, or swap in the ungated mirror `unsloth/Llama-3.2-3B-Instruct`.

### Imports

This notebook uses:

- Unsloth `FastLanguageModel` for memory-efficient model loading and LoRA
- vLLM (via Unsloth's `fast_inference` path) for high-throughput generation during training
- TRL `GRPOConfig` and `GRPOTrainer` for the GRPO training loop
- `datasets` for loading GSM8K

> NOTE: `UNSLOTH_VLLM_STANDBY` must be set **before** importing Unsloth — it lets training and vLLM share GPU memory instead of splitting it, freeing roughly 30% more context headroom.

In [1]:
import os

os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # must be set before importing unsloth

from unsloth import FastLanguageModel
import torch

assert torch.cuda.is_available(), "This notebook requires an NVIDIA GPU."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1e9:.1f} GB | Compute: {gpu.major}.{gpu.minor}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/alex/projects/code/The-AI-Engineering-Certification-v1.0/15_Reasoning_Model_Fine_Tuning/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: NVIDIA GeForce RTX 3080 | VRAM: 10.7 GB | Compute: 8.6


In [2]:
import getpass

from huggingface_hub import get_token, login

if get_token() is None:
    login(token=getpass.getpass("Hugging Face token (with the Llama 3.2 license accepted): "))

LocalTokenNotFoundError: Token is required (`token=True`), but no token found. You need to provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

## Task 2: Load the Base Model

Unsloth wraps Hugging Face model loading with `FastLanguageModel.from_pretrained`. The arguments that matter here:

- `max_seq_length = 2048` — the total budget for prompt + completion. Reasoning traces are long, so GRPO wants headroom; increase this if you train for longer traces.
- `load_in_4bit = False` — we deliberately train LoRA in **16-bit**. GRPO spends most of its wall-clock generating candidate completions, and vLLM's fast-generation path is fastest and most stable on 16-bit weights. Set this to `True` (QLoRA) only if you need to squeeze under ~8 GB of VRAM.
- `fast_inference = True` — runs the vLLM engine inside Unsloth so group sampling doesn't crawl.
- `max_lora_rank = 64` — must be at least the LoRA rank we pick in Task 3.
- `gpu_memory_utilization = 0.7` — the fraction of VRAM vLLM may claim for weights + KV cache. `0.7` fits a 16 GB card; raise toward `0.9` on 24 GB.

### A quick word on quantization (and why we skip it here)

Quantization discretizes weights from a representation that holds more information into one that holds less. The [QLoRA paper](https://arxiv.org/abs/2305.14314) made 4-bit fine-tuning practical with two tricks: the **NF4** data type (near-optimal for normally distributed weights, applied block-wise with one scaling constant per 64-weight block) and **double quantization** (quantizing the quantization constants themselves, saving ~0.37 bits per parameter). That is the right tool when GPU memory is the binding constraint. Here, generation throughput is the binding constraint — so we keep the weights in 16-bit and let LoRA keep the *trainable* footprint small instead.

In [2]:
max_seq_length = 2048  # can increase for longer reasoning traces
lora_rank = 64  # larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name = "meta-llama/Llama-3.2-3B-Instruct",  # or "unsloth/Llama-3.2-3B-Instruct" (ungated mirror)
    model_name = "unsloth/Llama-3.2-3B-Instruct", 
    max_seq_length = max_seq_length,
    load_in_4bit = True,  # 16-bit LoRA — see the Task 2 notes
    fast_inference = True,  # enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.6,  # raise toward 0.9 on a 24 GB card
)

WARNING 07-24 15:13:30 [interface.py:470] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
INFO 07-24 15:13:31 [vllm_utils.py:739] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.7.4: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.15.1.
   \\   /|    NVIDIA GeForce RTX 3080. Num GPUs = 1. Max memory: 10.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standby mode is enabled. Changing `gpu_memory_utilization` to 0.7124999999999999.
Unsloth: FlashInfer requires JIT compilation but nvcc (CUDA compiler) is not found.
  vLLM will use FLASH_ATTN attention + PyTorch sampler instead (works fine).
  To enable FlashInfer, install the missing tools:
    nvcc  - 

/home/alex/projects/code/The-AI-Engineering-Certification-v1.0/15_Reasoning_Model_Fine_Tuning/.venv/lib/python3.13/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:181: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
We recommend installing via `pip install torch-c-dlpack-ext`
  warnings.warn(


Unsloth: Not an error, but `use_cudagraph` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `use_inductor` is not supported in vLLM.config.CompilationConfig. Skipping.
WARNING 07-24 15:13:40 [compilation.py:762] Level is deprecated and will be removed in the next release,either 0.12.0 or 0.11.2 whichever is soonest.Use mode instead.If both level and mode are given,only mode will be used.
Unsloth: Not an error, but `device` is not supported in vLLM. Skipping.
INFO 07-24 15:13:40 [utils.py:261] non-default args: {'load_format': 'bitsandbytes', 'dtype': torch.bfloat16, 'max_model_len': 2048, 'enable_prefix_caching': True, 'swap_space': 0, 'gpu_memory_utilization': 0.6289650373553396, 'max_num_batched_tokens': 2048, 'max_num_seqs': 16, 'max_logprobs': 0, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'enable_lora': True, 'max_lora_rank': 64, 'enable_chunked_prefill': True, 'compilation_config': {'level': 3, 'mode': 3, 'debug_dump_path': No

/home/alex/projects/code/The-AI-Engineering-Certification-v1.0/15_Reasoning_Model_Fine_Tuning/.venv/lib/python3.13/site-packages/pydantic/type_adapter.py:607: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


WARNING 07-24 15:13:40 [arg_utils.py:1220] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 07-24 15:13:42 [model.py:541] Resolved architecture: LlamaForCausalLM
INFO 07-24 15:13:42 [model.py:1561] Using max model len 2048


...

INFO 07-24 15:14:07 [core.py:272] init engine (profile, create kv cache, warmup model) took 18.63 seconds
INFO 07-24 15:14:08 [llm.py:343] Supported tasks: ('generate',)


`torch_dtype` is deprecated! Use `dtype` instead!


Unsloth: Just some info: will skip parsing ['post_attention_layernorm', 'pre_feedforward_layernorm', 'norm1', 'k_norm', 'ffn_norm', 'norm', 'post_per_layer_input_norm', 'q_norm', 'layer_norm2', 'attention_norm', 'post_layernorm', 'input_layernorm', 'layer_norm1', 'post_feedforward_layernorm', 'norm2']


Some weights of LlamaForCausalLM were not initialized from the model checkpoint at unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['post_attention_layernorm', 'pre_feedforward_layernorm', 'norm1', 'k_norm', 'ffn_norm', 'norm', 'post_per_layer_input_norm', 'q_norm', 'layer_norm2', 'attention_norm', 'cross_attn_post_attention_layernorm', 'post_layernorm', 'input_layernorm', 'layer_norm1', 'post_feedforward_layernorm', 'norm2', 'cross_attn_input_layernorm']


## Task 3: Attach LoRA Adapters

Fine-tuning all 3B parameters is prohibitively expensive. [LoRA](https://arxiv.org/abs/2106.09685) — a PEFT (parameter-efficient fine-tuning) method — freezes the base weights and learns a low-rank update instead: for a weight matrix `W`, it trains two small matrices `A` and `B` of rank `r` such that the effective weight becomes `W + BA`. Only `A` and `B` receive gradients.

The knobs:

- `r = 64` — the rank of the decomposition. Higher rank = more trainable capacity (and more VRAM and time). Common values: 8, 16, 32, 64, 128.
- `lora_alpha = 64` — a scaling factor on the update; setting `alpha = r` is a common, stable default.
- `target_modules` — which weight matrices get adapters. The original LoRA paper adapted only attention weights; the QLoRA paper found adapting **all** linear layers works better, so we take the four attention projections plus the three MLP projections.
- `use_gradient_checkpointing = "unsloth"` — Unsloth's checkpointing variant, tuned for long-context RL training.
- `random_state = 3407` — reproducible adapter initialization.

The `print(model)` output below shows exactly where the `lora_A`/`lora_B` pairs were injected — you will need it for Question #2.

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth",  # enables long-context fine-tuning
    random_state = 3407,
)

print(model)

Unsloth 2026.7.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
  

#### ❓ Question #1

The [QLoRA paper](https://arxiv.org/abs/2305.14314) introduces *double quantization*. In your own words: what gets quantized the second time, and roughly how much memory does it save per parameter? This notebook sets `load_in_4bit = False` and trains a 16-bit LoRA instead — why is 16-bit the better fit for GRPO with vLLM fast inference, and when would you still reach for QLoRA?

##### Answer:

- When the weights are quantized to 4 bits with QLoRA (first quantization), a quantization constant is also saved blocks of 64 weights each. This quantization constant, which is 127/absmax(weight in a block), is saved in 32 bits. This means 32/(64-per-block)= 0.5 bits per weight overhead. When quantizing this constant too (second quantization), to 8 bits, it becomes 0.125 bits per weight overhead, meaning a 0.375 bits ovehead saved per weight. for 3.2B weights that's just 150MB (mega bytes) of memory (3.2B*0.375/8), but for a 65B model like in the QLoRA paper, that would be 3GB memory saved and that's significant when trying to fine tune on a single GPU.
- Quantizing the weights in 4 bits instead of loading them in memory in 16 bits (full base model) saves 4.8GB memory when loading the model, at the cost of a very small drop in performance when fine tuning the model, and computation complexity (for each LORA forward pass, each 4-bit weight needs to be computed back to its 16 bit form, using dequantization and the quantization constant). I actually ran this notebook on my 10GB memory GPU, and I needed to use load_in_4bit = True, and succesfully fined-tuned the model, and observed how in task 7 (training) it reached its "Aha" moment after roughly 60 steps, and the training with GPRO took me about 50 minutes, and the max memory load on the gpu was about 9GB, so I had to use 4-bit loading with QLoRA to save this memory. When there is no memory bottleneck, GPRO with vLLM is bottlenecked only by throughput, so If I had 5GB more memory, I could load the weights without quantization and the training would be significantly faster. Even though this notebook mentions that 16-bit is more stable on vLLM's fast inference path, i didn't find any stability issues. 

#### ❓ Question #2

![Transformer decoder diagram](https://i.imgur.com/N8y2crZ.png)

Using the `print(model)` output from Task 3, label the diagram with the matching layers from `meta-llama/Llama-3.2-3B-Instruct`'s architecture.

- EXAMPLE — Layer Norm:
  - `(input_layernorm): LlamaRMSNorm()`
  - `(post_attention_layernorm): LlamaRMSNorm()`
  - `(norm): LlamaRMSNorm()`
- Feed Forward:
- Masked Multi Self-Attention:
- Text & Position Embed:
- Text Prediction:

##### Answer:

- Feed Forward:
  - (mlp): LlamaMLP(...)
- Masked Multi Self-Attention:
  - (self_attn): LlamaAttention(...)
- Text & Position Embed:
  - (embed_tokens): Embedding(128256, 3072, padding_idx=128004)  # text embed
  - (rotary_emb): LlamaRotaryEmbedding()     # position embed
- Text Prediction:
  - (lm_head): Linear(in_features=3072, out_features=128256, bias=False)

#### ❓ Question #3

What, in your own words, is LoRA doing?

##### Answer:

LoRA is representing the adjustment of weights needed to fine-tune a model with an A*B matrix (applied on each forward pass when A and B are adjusted during training) whose multiplication is a matrix of the same shape and size as the model's original matrix so it can be added to the frozen weights matrix.
Because A and B are matrices of lower rank, and only A and B's weights are adjusted, effectively only a small fraction of the weights needs to be adjusted while training, saving the need to adjust individually each of the weights of the original model, whose weights are frozen. 

## Breakout Room #1 Summary

- The base model loads once in 16-bit, with vLLM (`fast_inference = True`) ready for high-throughput group sampling.
- LoRA freezes the base weights and trains low-rank `A`/`B` updates on all seven projection matrices — a small fraction of the total parameters.
- 16-bit LoRA is the right default for GRPO; QLoRA (4-bit) trades generation speed for memory when VRAM is the constraint.

---
# Breakout Room #2
## GRPO Training on GSM8K

GRPO turns fine-tuning into a reward game. Each step:

1. **Group sampling** — for a single prompt, the policy generates a *group* of completions (8 here) instead of just one.
2. **Reward scoring** — each completion is scored by our reward functions.
3. **Group-based advantage** — each completion's reward is compared to the group average: above average means a positive advantage, below means negative.
4. **Policy update** — the policy is nudged toward positive-advantage outputs, with a KL penalty preventing drastic drift.
5. **Iterate** — the updated policy samples the next groups, and the loop repeats until reward converges.

The group baseline replaces the separate value network (critic) that PPO-style RLHF needs — and no human preference data or process reward model is involved. All we need is a dataset whose answers we can *verify*.

## Task 4: Prepare the GSM8K Dataset

Notice something odd about the data: it is just questions and final answers — no reasoning traces, no preference pairs. That is the point. We never show the model *how* to reason; we only need a way to check whether an answer is right so we can reward it.

[GSM8K](https://huggingface.co/datasets/openai/gsm8k) marks each gold answer after a `####` delimiter, which makes verification a string comparison.

To make completions *checkable*, the system prompt forces a structure we can parse:

```text
<reasoning> ... </reasoning>
<answer> ... </answer>
```

`extract_xml_answer` pulls the model's answer out of that structure; `extract_hash_answer` pulls the gold answer out of GSM8K's `####` format.

> NOTE: This is not exactly the DeepSeek-R1 recipe — R1 adds a small SFT "cold start" stage to prime the model before RL. We skip straight to RL. The data prep and reward functions below build directly on [@willccbb's gist](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb), by way of Unsloth.

In [4]:
import re

from datasets import Dataset, load_dataset

SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    """Return the text between the <answer> and </answer> tags."""
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    """Return the gold answer after GSM8K's '####' marker, or None if absent."""
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

def get_gsm8k_questions(split="train") -> Dataset:
    """Load GSM8K and map each item to a system+user prompt plus the extracted gold answer."""
    data = load_dataset("openai/gsm8k", "main")[split]
    data = data.map(lambda x: {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": x["question"]},
        ],
        "answer": extract_hash_answer(x["answer"]),
    })
    return data

dataset = get_gsm8k_questions()

In [5]:
dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': '72',
 'prompt': [{'role': 'system',
   'content': '\nRespond in the following format:\n<reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer>\n'},
  {'role': 'user',
   'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'}]}

## Task 5: Define Reward Functions

Here is the *magic* of the approach: instead of showing the model examples of good reasoning, we score its attempts with a stack of reward functions and let GRPO figure out the rest.

![Reward function stack](https://i.imgur.com/7Dp0qdt.png)

| Reward function | What it checks | Max reward |
|---|---|---|
| `correctness_reward_func` | the extracted answer equals the gold answer | 2.0 |
| `int_reward_func` | the extracted answer is a plain integer | 0.5 |
| `strict_format_reward_func` | exact `<reasoning>`/`<answer>` layout, newlines and all | 0.5 |
| `soft_format_reward_func` | the tags appear in the right order (lenient) | 0.5 |
| `xmlcount_reward_func` | partial credit per correctly placed tag, minus a trailing-junk penalty | ~0.5 |

Correctness dominates (2.0 vs 0.5), but the format rewards give the model a *gradient to climb* early on — a completion can earn partial credit for structure before it ever gets an answer right. These functions are fully customizable: they are how you steer what the model is incentivized to get good at.

In [6]:
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """2.0 if the extracted <answer> matches the gold answer, else 0.0.

    Also prints the first question/response pair of each batch so you can watch
    completions evolve from rambling answers into structured reasoning.
    """
    responses = [completion[0]["content"] for completion in completions]
    q = prompts[0][-1]["content"]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print(
        "-" * 20,
        f"Question:\n{q}",
        f"\nGold answer:\n{answer[0]}",
        f"\nResponse:\n{responses[0]}",
        f"\nExtracted:\n{extracted_responses[0]}",
        sep="\n",
    )
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    """0.5 if the extracted answer is a plain integer, else 0.0."""
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

In [7]:
def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """0.5 if the completion matches the exact expected layout, newlines and all."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    # re.DOTALL lets .*? span newlines — without it, multi-line reasoning never matches
    matches = [re.match(pattern, r, flags=re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """0.5 if <reasoning> and <answer> blocks appear in order, however loosely."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, flags=re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

In [8]:
def count_xml(text) -> float:
    """Partial credit (0.125 each) per correctly placed tag; small penalty for trailing junk."""
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1]) * 0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

In [9]:
# Quick sanity check: soft accepts a single-line completion, strict wants each tag on its own line.
sample = [[{"content": "<reasoning>Wow, cool!</reasoning><answer>23</answer>"}]]
print("soft:  ", soft_format_reward_func(sample))
print("strict:", strict_format_reward_func(sample))

soft:   [0.5]
strict: [0.0]


## Task 6: Configure GRPO

With training examples and reward functions in hand, all that is left is configuration. Two batch-geometry rules to understand first:

- `num_generations = 8` — the *group size* GRPO compares within. Decrease to 4 if you run out of memory (noisier advantages, less VRAM).
- TRL requires that `per_device_train_batch_size × gradient_accumulation_steps` be **divisible by `num_generations`** — the sampled groups must tile evenly into the effective batch. That is why we use `gradient_accumulation_steps = 8` here.

The rest are familiar fine-tuning knobs (`learning_rate`, `warmup_ratio`, `lr_scheduler_type` — see Question #4), plus:

- `max_prompt_length` / `max_completion_length` — we measure the longest tokenized prompt in the dataset and give completions everything that remains of `max_seq_length`.
- `optim = "adamw_8bit"` — 8-bit optimizer states shave a couple of GB of VRAM.
- `max_steps = 175` — enough to see the reward curve turn upward within a class session; for a full training run, train for an epoch or more.
- `report_to = "none"` — flip to `"wandb"` for the classic "line goes up and to the right" chart (install with `uv sync --extra wandb`, then `wandb login`).

In [11]:
max_prompt_length = max(dataset.map(
    lambda x: {"tokens": tokenizer.apply_chat_template(x["prompt"], add_generation_prompt=True, tokenize=True)},
    batched=True,
).map(lambda x: {"length": len(x["tokens"])})["length"])

max_prompt_length = max_prompt_length + 1  # +1 just in case!
print(f"Longest prompt: {max_prompt_length} tokens")

Longest prompt: 267 tokens


In [12]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    learning_rate = 5e-6,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,  # batch_size x grad_accum must be divisible by num_generations
    num_generations = 8,  # group size; decrease to 4 if out of memory (and set grad_accum to 4)
    max_prompt_length = max_prompt_length,
    max_completion_length = max_seq_length - max_prompt_length,
    max_steps = 175,
    save_steps = 25,
    max_grad_norm = 0.1,
    report_to = "none",  # flip to "wandb" for experiment tracking
    output_dir = "outputs",
)

#### ❓ Question #4

Describe what the following parameters are doing:

- `warmup_ratio`
- `learning_rate`
- `lr_scheduler_type`

> NOTE: Feel free to consult the [TrainingArguments documentation](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments) or other resources!

##### Answer:

- `warmup_ratio`: Spend 10% of the training time gradually increasing the actual step size (by how much each weight can be moved in a direction) to the max defined step size, which is 'learning_rate'.
- `learning_rate`: As already explained in 'warmup_ratio', this is the maximal change each of the weights can be "nudged" by during training for each step. 
- `lr_scheduler_type`: After warmup is over, and the maximal step size is reached (learning_rate), this parameter defines whether and how step size changes. When set to "cosine" as in our example above, it will gradually shrink in a cosine shaped curve:

![cosine scheduler](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/warmup_cosine_schedule.png)


as we can see in the figure above, once the step size was increased from 0 to the max (learning_rate) as defined by the ratio, it gradually decreases so that once it reaches the sweet spot it won't overshoot it. . 


## Task 7: Train with GRPOTrainer

Unlike supervised fine-tuning, where you watch loss go *down*, here you watch reward go *up*. What to expect:

- Each logged step shows one column per reward function plus the combined `reward` — you can see the format rewards kick in before correctness does.
- The debug print in `correctness_reward_func` lets you watch raw completions evolve in real time.
- There is a famous "Aha!" moment: reward hovers near ~0 and then suddenly starts climbing. Expect little movement before step ~100–150, and budget roughly an hour at these settings. (Unsloth recommends 300+ steps for clearly strong results — 175 keeps this class-sized.)

Here is what the reward curve looked like on a previous run of this notebook — noisy at this scale, but unmistakably up and to the right:

![Example reward curve](https://i.imgur.com/7sBk5y2.png)

In [13]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1 | Total steps = 175
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 97,255,424 of 3,310,005,248 (2.94% trained)


WARNING 07-24 15:15:20 [input_processor.py:287] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!
--------------------
Question:
A concert ticket costs $40. Mr. Benson bought 12 tickets and received a 5% discount for every ticket bought that exceeds 10. How much did Mr. Benson pay in all?

Gold answer:
476

Response:
<reasoning>
To find the total cost, we first need to calculate the cost of the first 10 tickets, which are full price, and then the cost of the 2 remaining tickets, which are discounted.
Since Mr. Benson bought 12 tickets and gets a 5% discount for every ticket exceeding 10, the first 10 tickets will be full price, and the remaining 2 tickets will receive a 5% discount.
We will then calculate the total cost by adding the 

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / xmlcount_reward_func / mean,rewards / xmlcount_reward_func / std,rewards / soft_format_reward_func / mean,rewards / soft_format_reward_func / std,rewards / strict_format_reward_func / mean,rewards / strict_format_reward_func / std,rewards / int_reward_func / mean,rewards / int_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,-0.000000,-0.292625,0.448622,263.625000,188.000000,376.000000,0.000000,263.625000,188.000000,376.000000,-0.000000,-0.355125,0.428945,0.062500,0.176777,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,-0.615250,0.637696,365.250000,189.000000,500.000000,0.000000,365.250000,189.000000,500.000000,0.000458,-0.802750,0.562722,0.187500,0.258775,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,-0.225000,0.362748,221.875000,138.000000,328.000000,0.000000,221.875000,138.000000,328.000000,0.000395,-0.350000,0.403574,0.125000,0.231455,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,-0.071125,0.226657,238.750000,148.000000,309.000000,0.000000,238.750000,148.000000,309.000000,0.000453,-0.071125,0.226657,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.000000,-0.014375,0.062312,116.625000,76.000000,190.000000,0.000000,116.625000,76.000000,190.000000,0.000865,-0.014375,0.062312,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.000000,0.085875,0.268003,236.125000,136.000000,317.000000,0.000000,236.125000,136.000000,317.000000,0.000722,0.023375,0.160981,0.000000,0.000000,0.000000,0.000000,0.062500,0.176777,0.000000,0.000000
7,0.000000,-0.248375,0.178894,195.000000,130.000000,253.000000,0.000000,195.000000,130.000000,253.000000,0.000459,-0.248375,0.178894,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.264750,0.343155,146.000000,70.000000,194.000000,0.000000,146.000000,70.000000,194.000000,0.000582,-0.047750,0.198490,0.312500,0.258775,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,0.000000,0.754125,1.177282,196.000000,100.000000,274.000000,0.000000,196.000000,100.000000,274.000000,0.000520,-0.308375,0.175186,0.250000,0.267261,0.000000,0.000000,0.312500,0.258775,0.500000,0.925820
10,0.000000,-0.395000,0.342717,233.250000,153.000000,287.000000,0.000000,233.250000,153.000000,287.000000,0.000501,-0.395000,0.342717,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


--------------------
Question:
Jane is trying to decide whether to buy a house or a trailer. A house costs $480,000 and a trailer costs $120,000. Each loan will be paid in monthly installments over 20 years. How much more is the monthly payment on the house compared to the trailer?

Gold answer:
1500

Response:
<reasoning>
Let's break it down step by step. 

First, we need to find the total amount paid for each purchase over 20 years. 

For the house, the payment over the 20 years is equal to the total cost divided by 12 (for the months) and then multiplied by a common factor that's between 0 and 1. That factor represents the equivalent of paying in equal annual percentage rate on the present upfront payment. The formula for that factor is (1 + r/n)^(n*Years). Here, 'r' is the annual interest rate and 'n' is the number of periods in a year, which in our case is 12 since there are 12 months in a year. However, we'll simplify our calculation if we think of a loan not having an interest r

TrainOutput(global_step=175, training_loss=6.260566972481944e-05, metrics={'train_runtime': 1930.0433, 'train_samples_per_second': 0.725, 'train_steps_per_second': 0.091, 'total_flos': 0.0, 'train_loss': 6.260566972481944e-05})

## Task 8: Compare Before and After

Time for the payoff. A subtle but important detail: vLLM is still holding the *frozen base weights* — LoRA never touched them. That means we can generate from the plain base model at any time by passing `lora_request = None`, and from our trained model by passing the saved adapter. Same GPU, same engine, two personalities.

First, the base model — no system prompt, no LoRA:

In [14]:
from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)

text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Calculate pi."}],
    tokenize = False,
    add_generation_prompt = True,
)

base_output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,  # frozen base weights only
)[0].outputs[0].text

print(base_output)

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.27s/it, est. speed input: 30.74 toks/s, output: 133.97 toks/s]

I can calculate pi for you.

Pi (π) is a mathematical constant representing the ratio of a circle's circumference to its diameter. It is approximately equal to 3.14159.

If you want a more precise calculation, I can use the Bailey–Borwein–Plouffe (BBP) formula to calculate pi to a certain number of decimal places.

For example, here's pi calculated to 50 decimal places using the BBP formula:

3.14159265358979323846264338327950288419716939937511

Keep in mind that pi is an irrational number, which means it cannot be expressed exactly as a finite decimal or fraction. The more decimal places you calculate, the more digits of pi you'll get.

Would you like me to calculate pi to a specific number of decimal places?


## Task 9: Save and Load the LoRA

`save_lora` writes only the adapter (the `A`/`B` matrices) — a few hundred megabytes at rank 64, versus multiple gigabytes for full weights. We then reload it as a `lora_request` and rerun the same prompt, this time with the reasoning system prompt the model was trained against:

In [15]:
model.save_lora("grpo_saved_lora")

In [16]:
text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Calculate pi."},
    ],
    tokenize = False,
    add_generation_prompt = True,
)

trained_output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

print(trained_output)

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.51s/it, est. speed input: 40.38 toks/s, output: 80.75 toks/s]

<reasoning>
To calculate pi, we will be using the Leibniz formula for pi, which is a method for approximating pi using the sum of an infinite series. This formula is given by:

π = 4 * (1 - 1/3 + 1/5 - 1/7 + 1/9 - ...)

We will approximate pi by summing the first n terms of this series, where n is a large number.
</reasoning>
<answer>
3.141592653589793238462643383279502884197
</answer>


## Breakout Room #2 Summary

- GSM8K needs no reasoning traces or preference data — verifiable final answers are enough for RL.
- Stacked reward functions shape behavior: cheap format rewards provide an early gradient, and correctness (2.0) dominates once the model starts solving problems.
- GRPO's group-relative advantage (a group of 8 here) replaces PPO's value network; TRL enforces that the effective batch tiles evenly into groups.
- Training shows a delayed "Aha!" — flat reward, then liftoff — rather than a smoothly descending loss.
- The trained artifact is just a LoRA adapter: swap it in and out of the same vLLM engine via `lora_request`.

Where to go next: train longer (300+ steps), try a larger base model, write your own reward functions, or export merged 16-bit weights for serving — the [Unsloth documentation](https://docs.unsloth.ai/) covers all of the above.